# Step 3 — Audit and clean reference-award author attributes

**Inputs:** frozen award cohort v1, provisional reference-award outputs v1, existing author profiles, and optionally the cached author works.

**Outputs:** one provisional attribute row per identified author, positions for every reference-year award paper, paper affiliations in a separate long table, first-publication evidence, and review queues.

## Limits
- No edits to frozen data or step-2 outputs. No control data, citation metrics, or reference-event reselection.
- No live API calls. No API key needed. Existing first-publication years are not accepted simply because they are populated.
- Earliest cached work = earliest observed indexed work in that cache, not independently verified career start. It can still reflect author-ID conflation, erroneous dates, or incomplete pagination.
- Clean seniority means passing this notebook's stated basic data checks, **not manually validated identity**. Original values, candidate values, review reasons and evidence are retained separately.
- Paper affiliations describe the reference paper; they are not current author-profile affiliations and are not independent confirmation of employment on the award date.
- The representative paper for same-year ties is a convenience only. All reference-year papers remain in the long outputs.

Suggested placement: `notebooks/03_clean_author_attributes.ipynb`.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import pickle
import re
import shutil
import tempfile
import pandas as pd
from IPython.display import display

ROOT_OVERRIDE = None
FREEZE_REL = 'conf_paper/conf_data/frozen/award_cohort_v1'
STEP2_REL = 'conf_paper/conf_data/derived/reference_awards_v1'
PROFILES_REL = 'conf_paper/conf_data/02_author_profiles.csv'
CACHE_REL = 'conf_paper/conf_data/04_all_works_cache.pkl'
USE_WORKS_CACHE = True
OUTPUT_VERSION = 'author_attributes_v1'
WRITE_OUTPUTS = True
EARLY_YEAR_REVIEW_CUTOFF = 1940  # heuristic only, NOT an automatic date correction
MAX_CAREER_AGE_REVIEW = 60      # heuristic only; genuine long careers may exist
JUNIOR_CUTOFF = 5
TRANSFORM_VERSION = 'author_attributes_1.0'

def digest_bytes(b):
    return hashlib.sha256(b).hexdigest()

def digest(p):
    return digest_bytes(p.read_bytes())

def csv(p):
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig')

def find_root():
    if ROOT_OVERRIDE:
        root=Path(ROOT_OVERRIDE).expanduser().resolve()
        if not (root / STEP2_REL / 'manifest.json').is_file():
            raise FileNotFoundError('Configured root has no step-2 manifest.')
        return root
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / STEP2_REL / 'manifest.json').is_file():
            return p
    raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')

def check_outputs(folder):
    m=json.loads((folder/'manifest.json').read_text(encoding='utf-8'))
    for name,h in m['output_sha256'].items():
        if not (folder/name).is_file() or digest(folder/name)!=h:
            raise RuntimeError('Integrity failure: '+str(folder/name)+'. Investigate edits or line-ending changes; do not bypass.')
    return m

def boolean(s):
    vals=s.fillna('').astype(str).str.strip().str.lower()
    if not set(vals).issubset({'true','false',''}):
        raise ValueError('Unrecognized boolean column.')
    return vals.eq('true')

def norm_id(value, prefix='A'):
    s=str(value or '').strip()
    m=re.fullmatch(r'(?:https?://openalex\.org/)?('+prefix+r'\d+)/?',s,re.I)
    return m.group(1).upper() if m else None

def year_int(value):
    try:
        n=float(value)
        return int(n) if n.is_integer() and 1<=n<=9999 else None
    except (ValueError, TypeError, OverflowError):
        return None

def list_json(value):
    if value is None or str(value).strip() in {'','null','None','nan'}:
        return [], 'missing'
    try:
        parsed=json.loads(str(value))
    except (ValueError,TypeError):
        return [], 'parse_error'
    if not isinstance(parsed,list):
        return [], 'not_a_list'
    return parsed, 'empty' if not parsed else 'parsed'

ROOT=find_root(); FROZEN=ROOT/FREEZE_REL; STEP2=ROOT/STEP2_REL
TARGET=ROOT/'conf_paper/conf_data/derived'/OUTPUT_VERSION
freeze_manifest=check_outputs(FROZEN)
step2_manifest=check_outputs(STEP2)
for name,h in step2_manifest['identity']['input_hashes'].items():
    if digest(FROZEN/name)!=h:
        raise RuntimeError('Step 2 is not linked to the current freeze: '+name)
roster=csv(STEP2/'author_reference_roster.csv')
refs=csv(STEP2/'reference_year_award_links.csv')
papers=csv(FROZEN/'award_cohort_papers.csv')
assert roster.author_id.is_unique
assert refs[['author_id','cohort_record_id']].duplicated().sum()==0
assert set(refs.author_id)==set(roster.author_id)
assert set(refs.cohort_record_id).issubset(papers.cohort_record_id)
refs['award_year']=refs.award_year.map(year_int)
roster['reference_award_year']=roster.reference_award_year.map(year_int)
assert refs.award_year.notna().all() and roster.reference_award_year.notna().all()
# Check each reference paper's award year against the author's roster year.
expected_year = refs["author_id"].map(
    roster.set_index("author_id")["reference_award_year"]
)

assert expected_year.notna().all(), (
    "Some reference-paper authors are missing from the author roster."
)

assert refs["award_year"].eq(expected_year).all(), (
    "Some reference papers do not match the author's reference award year."
)
input_hashes={'freeze_manifest':digest(FROZEN/'manifest.json'), 'step2_manifest':digest(STEP2/'manifest.json')}
print('Verified frozen and step-2 artifacts. Identified authors:',len(roster))

Verified frozen and step-2 artifacts. Identified authors: 2710


## 1. Profile years and safely read the cached works

The cache loader forbids all pickle global/class imports and persistent references. It accepts plain built-in dictionary/list data only and does not call `pickle.load`. Skip the cache if it fails this restriction; do not switch to unrestricted loading. Only load files from your trusted repository.

The cache can contain repeated co-authored works across authors. Count and inspect evidence separately for each author ID. A cache/profile work-count mismatch is a coverage warning, not conclusive evidence of a data error or complete/incomplete collection.

In [3]:
if (ROOT/PROFILES_REL).exists():
    profiles=csv(ROOT/PROFILES_REL)
    input_hashes['profiles']=digest(ROOT/PROFILES_REL)
    profiles['author_id_norm']=profiles.author_id.map(norm_id)
    profile_duplicate_ids=profiles.loc[profiles.author_id_norm.notna() & profiles.author_id_norm.duplicated(keep=False)].copy()
    profile_lookup={k:g for k,g in profiles.loc[profiles.author_id_norm.notna()].groupby('author_id_norm')}
else:
    profiles=pd.DataFrame();profile_duplicate_ids=pd.DataFrame();profile_lookup={}

class BuiltinsOnlyUnpickler(pickle.Unpickler):
    def find_class(self,module,name):
        raise pickle.UnpicklingError('Global/class loading is forbidden.')
    def persistent_load(self,pid):
        raise pickle.UnpicklingError('Persistent references are forbidden.')

works_lookup={}; cache_status='disabled'; cache_failed_ids=set(); cache_key_conflicts=set()
if USE_WORKS_CACHE and (ROOT/CACHE_REL).exists():
    cache_bytes=(ROOT/CACHE_REL).read_bytes()
    input_hashes['works_cache']=digest_bytes(cache_bytes)
    try:
        cache=BuiltinsOnlyUnpickler(io.BytesIO(cache_bytes)).load()
        if not isinstance(cache,dict) or not isinstance(cache.get('all_works'),dict):
            raise ValueError('Unexpected cache schema.')
        for aid,works in cache['all_works'].items():
            key=norm_id(aid)
            if key:
                if key in works_lookup:
                    cache_key_conflicts.add(key)
                else:
                    works_lookup[key]=works
        for item in cache.get('failed_authors',[]):
            aid=item[0] if isinstance(item,(list,tuple)) and item else item
            key=norm_id(aid)
            if key:cache_failed_ids.add(key)
        cache_status='loaded_with_restricted_unpickler'
    except Exception as exc:
        works_lookup={};cache_status='unavailable: '+type(exc).__name__
elif USE_WORKS_CACHE:
    cache_status='file_missing'
print('Works cache status:',cache_status)
print('Normalized author keys in usable cache:',len(works_lookup))
print('Duplicate normalized cache keys:',len(cache_key_conflicts))
display(profile_duplicate_ids)

Works cache status: loaded_with_restricted_unpickler
Normalized author keys in usable cache: 2701
Duplicate normalized cache keys: 0


,author_id,display_name,works_count,cited_by_count,h_index,i10_index,first_pub_year_countsbyyear,first_pub_year,author_id_norm


## 2. First-publication evidence and conservative seniority labels

Preferred candidate = earliest dated, valid-ID work in the cached works. Profile-only dates remain unverified. Review flags include conflicting dates, invalid/very early years, negative/exceptionally long career ages, conflicting cache work years, failed cache collection, and insufficient cache evidence.

No implausibly old work is silently removed to choose a later, more convenient start year. `seniority_clean` stays **Unknown** for flagged cases. `seniority_candidate` and the original profile-based label remain visible for comparison. All labels use the step-2 reference year: Junior = career age 0–5 inclusive, Senior = >5.

In [4]:
NOW_YEAR=datetime.now(timezone.utc).year
firstpub_rows=[]; earliest_work_rows=[]
for row in roster.itertuples(index=False):
    aid=row.author_id; ref_year=int(row.reference_award_year); reasons=[]
    pg=profile_lookup.get(aid)
    profile_exists=pg is not None
    profile_duplicate=profile_exists and len(pg)>1
    profile_years=sorted({y for v in (pg['first_pub_year'] if profile_exists and 'first_pub_year' in pg else []) if (y:=year_int(v)) is not None})
    profile_year=profile_years[0] if len(profile_years)==1 else None
    pcount=None
    if profile_exists and len(pg)==1 and 'works_count' in pg:
        pcount=year_int(pg.iloc[0]['works_count'])
    if profile_duplicate:reasons.append('duplicate_profile_author_id')
    if len(profile_years)>1:reasons.append('conflicting_profile_first_years')
    works=works_lookup.get(aid)
    work_years={}; invalid_rows=0; year_conflicts=False; cache_shape_bad=False
    if works is not None:
        if not isinstance(works,list):
            cache_shape_bad=True
        else:
            for w in works:
                if not isinstance(w,dict):invalid_rows+=1;continue
                wid=norm_id(w.get('id'),'W');yr=year_int(w.get('publication_year'))
                if wid is None or yr is None or yr>NOW_YEAR:
                    invalid_rows+=1;continue
                work_years.setdefault(wid,set()).add(yr)
            year_conflicts=any(len(ys)>1 for ys in work_years.values())
    cache_year=min((y for ys in work_years.values() for y in ys),default=None)
    if cache_year is not None:
        for wid,ys in work_years.items():
            if cache_year in ys:
                earliest_work_rows.append({'author_id':aid,'work_id':wid,'observed_first_year':cache_year,
                    'evidence_source':'local_works_cache','identity_independently_verified':False})
    chosen=cache_year if cache_year is not None else profile_year
    provenance='earliest_cached_work' if cache_year is not None else ('profile_only' if profile_year is not None else 'missing')
    if chosen is None:reasons.append('no_valid_first_year')
    if cache_year is None:reasons.append('no_usable_works_cache_evidence')
    if cache_shape_bad:reasons.append('cache_author_entry_not_list')
    if invalid_rows:reasons.append('invalid_cached_work_ids_or_years')
    if year_conflicts:reasons.append('conflicting_years_for_same_cached_work')
    if aid in cache_key_conflicts:reasons.append('duplicate_normalized_cache_keys')
    if aid in cache_failed_ids:reasons.append('author_in_failed_cache_collection_log')
    if cache_year is not None and profile_year is not None and cache_year!=profile_year:
        reasons.append('cache_profile_first_year_disagreement')
    for label,y in [('candidate',chosen),('profile',profile_year)]:
        if y is not None:
            if y<EARLY_YEAR_REVIEW_CUTOFF:reasons.append(label+'_very_early_first_year')
            if y>ref_year:reasons.append(label+'_first_year_after_reference_award')
            if ref_year-y>MAX_CAREER_AGE_REVIEW:reasons.append(label+'_career_age_over_review_threshold')
            if y>NOW_YEAR:reasons.append(label+'_future_year')
    age=ref_year-chosen if chosen is not None else None
    candidate='Unknown' if age is None or age<0 else ('Junior' if age<=JUNIOR_CUTOFF else 'Senior')
    profile_age=ref_year-profile_year if profile_year is not None else None
    old_stage='Unknown' if profile_age is None or profile_age<0 else ('Junior' if profile_age<=JUNIOR_CUTOFF else 'Senior')
    reasons=sorted(set(reasons))
    clean=candidate if not reasons else 'Unknown'
    firstpub_rows.append({'author_id':aid,'reference_award_year':ref_year,
        'profile_record_available':profile_exists,'first_pub_year_profile':profile_year,
        'first_pub_year_cache':cache_year,'first_pub_year_candidate':chosen,
        'first_pub_year_evidence_source':provenance,'career_age_candidate':age,
        'seniority_profile_at_reference':old_stage,'seniority_candidate':candidate,
        'seniority_clean':clean,'first_pub_requires_review':bool(reasons),
        'first_pub_review_reasons':' | '.join(reasons),
        'first_pub_validation_status':'requires_review' if reasons else 'passed_basic_cache_checks_not_identity_validated',
        'n_unique_cached_works_with_valid_year':len(work_years),
        'n_invalid_cached_work_rows':invalid_rows,
        'profile_works_count_snapshot':pcount,
        'cache_vs_profile_count_differs':pcount is not None and len(work_years)!=pcount,
        'cache_pagination_completeness_verified':False,
        'author_identity_independently_verified':False})
firstpub=pd.DataFrame(firstpub_rows)
for col in ['first_pub_year_profile','first_pub_year_cache','first_pub_year_candidate','career_age_candidate','profile_works_count_snapshot']:
    firstpub[col]=firstpub[col].astype('Int64')
earliest_evidence=pd.DataFrame(earliest_work_rows,columns=['author_id','work_id','observed_first_year','evidence_source','identity_independently_verified'])
display(firstpub.seniority_clean.value_counts(dropna=False).rename_axis('seniority_clean').to_frame('authors'))
display(firstpub.loc[firstpub.first_pub_requires_review].head(30))

,authors
seniority_clean,
Senior,1723
Junior,776
Unknown,211


,author_id,reference_award_year,profile_record_available,first_pub_year_profile,first_pub_year_cache,first_pub_year_candidate,first_pub_year_evidence_source,career_age_candidate,seniority_profile_at_reference,seniority_candidate,seniority_clean,first_pub_requires_review,first_pub_review_reasons,first_pub_validation_status,n_unique_cached_works_with_valid_year,n_invalid_cached_work_rows,profile_works_count_snapshot,cache_vs_profile_count_differs,cache_pagination_completeness_verified,author_identity_independently_verified
18,A5000796744,2018,True,2008,2008,2008,earliest_cached_work,10,Senior,Senior,Unknown,True,invalid_cached_work_ids_or_years,requires_review,29,1,30,True,False,False
27,A5001232416,2018,True,1899,1899,1899,earliest_cached_work,119,Senior,Senior,Unknown,True,candidate_career_age_over_review_threshold | c...,requires_review,320,0,320,False,False,False
32,A5001534492,2008,True,1991,1991,1991,earliest_cached_work,17,Senior,Senior,Unknown,True,invalid_cached_work_ids_or_years,requires_review,268,1,269,True,False,False
58,A5002419188,2004,True,1899,1899,1899,earliest_cached_work,105,Senior,Senior,Unknown,True,candidate_career_age_over_review_threshold | c...,requires_review,326,0,326,False,False,False
65,A5002599291,2013,True,1934,1934,1934,earliest_cached_work,79,Senior,Senior,Unknown,True,candidate_career_age_over_review_threshold | c...,requires_review,253,0,253,False,False,False
74,A5003040843,2011,True,1879,1879,1879,earliest_cached_work,132,Senior,Senior,Unknown,True,candidate_career_age_over_review_threshold | c...,requires_review,652,1,653,True,False,False
104,A5004444958,2012,True,1998,1998,1998,earliest_cached_work,14,Senior,Senior,Unknown,True,invalid_cached_work_ids_or_years,requires_review,270,1,273,True,False,False
113,A5004988074,2013,True,2009,2009,2009,earliest_cached_work,4,Junior,Junior,Unknown,True,invalid_cached_work_ids_or_years,requires_review,29,1,30,True,False,False
117,A5005255629,2010,True,1905,1905,1905,earliest_cached_work,105,Senior,Senior,Unknown,True,candidate_career_age_over_review_threshold | c...,requires_review,191,0,191,False,False,False
133,A5005730180,2006,True,1986,1986,1986,earliest_cached_work,20,Senior,Senior,Unknown,True,invalid_cached_work_ids_or_years,requires_review,309,1,309,False,False,False


## 3. Extract original authorship order and all paper institutions

Keep the original list order including unidentified co-authors, so missing IDs do not shift ranks. Check the index against the frozen paper's parsed slot count and the stored first/middle/last label. This validates internal metadata consistency, not the original PDF.

For every reference-year award, preserve all affiliations, institution IDs/names/countries, and raw affiliation strings. A missing structured institution does not mean the author had no institution. Multi-affiliated authors produce multiple rows; papers with no structured institutions receive an explicit missing-affiliation row.

In [5]:
pcols=['cohort_record_id','n_structured_authorship_slots']
ref_attributes=refs.merge(papers[pcols],on='cohort_record_id',how='left',validate='many_to_one')
ref_attributes['author_order_clean']=pd.to_numeric(ref_attributes.author_order_in_source,errors='coerce').astype('Int64')
ref_attributes['n_authors_on_paper']=pd.to_numeric(ref_attributes.n_structured_authorship_slots,errors='coerce').astype('Int64')

def position_audit(row):
    order=row.author_order_clean;total=row.n_authors_on_paper
    if pd.isna(order) or pd.isna(total) or total<1 or order<1 or order>total:
        return 'Unknown','invalid_order_or_list_length'
    expected='first' if order==1 else ('last' if order==total else 'middle')
    source_label=str(row.author_position).strip().lower()
    allowed={expected}
    if total==1:allowed={'first','last'}
    return expected,('consistent_with_source_list' if source_label in allowed else 'source_position_label_disagreement')
pa=ref_attributes.apply(position_audit,axis=1)
ref_attributes['position_from_list']=[x[0] for x in pa]
ref_attributes['position_check_status']=[x[1] for x in pa]
ref_attributes['authorship_order_verified_against_pdf']=False

institution_rows=[];aff_flags=[]
for row in ref_attributes.to_dict('records'):
    institutions,status=list_json(row.get('institutions_json'))
    raw_affs,raw_status=list_json(row.get('raw_affiliation_strings_json'))
    bad_entries=any(not isinstance(x,dict) for x in institutions)
    valid_insts=[x for x in institutions if isinstance(x,dict)]
    if not valid_insts:valid_insts=[None]
    usable_count=0
    for index,inst in enumerate(valid_insts,1):
        inst=inst or {}
        iid=norm_id(inst.get('id'),'I');name=inst.get('display_name')
        populated=bool(iid or (isinstance(name,str) and name.strip()))
        usable_count+=int(populated)
        institution_rows.append({'author_id':row['author_id'],'cohort_record_id':row['cohort_record_id'],
            'reference_award_year':row['award_year'],'institution_slot':index,
            'institution_id':iid,'institution_id_raw':inst.get('id'),
            'institution_name':name,'country_code':inst.get('country_code'),
            'institution_type':inst.get('type'),'ror':inst.get('ror'),
            'structured_institution_missing':not populated,
            'paper_affiliation_parse_status':status,
            'raw_affiliation_strings_json':json.dumps(raw_affs,ensure_ascii=False),
            'raw_affiliation_parse_status':raw_status,
            'affiliation_evidence':'reference_award_paper_metadata',
            'award_date_employment_independently_verified':False})
    aff_flags.append({'author_id':row['author_id'],'cohort_record_id':row['cohort_record_id'],
        'n_populated_institution_entries':usable_count,
        'affiliation_requires_review':status not in {'parsed'} or bad_entries or usable_count==0,
        'affiliation_parse_status':status,'affiliation_has_malformed_entries':bad_entries,
        'raw_affiliation_parse_status':raw_status})
affiliations=pd.DataFrame(institution_rows)
ref_attributes=ref_attributes.merge(pd.DataFrame(aff_flags),on=['author_id','cohort_record_id'],validate='one_to_one')
display(ref_attributes[['author_id','cohort_record_id','author_order_clean','n_authors_on_paper',
                        'author_position','position_from_list','position_check_status',
                        'n_populated_institution_entries','affiliation_requires_review']].head(20))

,author_id,cohort_record_id,author_order_clean,n_authors_on_paper,author_position,position_from_list,position_check_status,n_populated_institution_entries,affiliation_requires_review
0,A5000009841,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,6,middle,middle,consistent_with_source_list,1,False
1,A5000102893,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,4,first,first,consistent_with_source_list,1,False
2,A5000107991,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,8,middle,middle,consistent_with_source_list,1,False
3,A5000163896,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,2,first,first,consistent_with_source_list,1,False
4,A5000167011,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,4,4,last,last,consistent_with_source_list,1,False
5,A5000170074,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,7,middle,middle,consistent_with_source_list,1,False
6,A5000245150,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,4,4,last,last,consistent_with_source_list,2,False
7,A5000246567,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,2,first,first,consistent_with_source_list,2,False
8,A5000294737,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,4,5,middle,middle,consistent_with_source_list,5,False
9,A5000308912,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,4,middle,middle,consistent_with_source_list,2,False


## 4. Combine one row per author, retaining tied-paper detail

Representative fields come from the step-2 representative paper. The JSON and long tables retain all same-year award-paper positions and affiliations, avoiding silent loss for the 27 tie cases found in the reviewed step-2 output. The notebook computes current counts rather than assuming that number.

In [6]:
rep_keys=roster[['author_id','representative_cohort_record_id']].rename(columns={'representative_cohort_record_id':'cohort_record_id'})
rep=rep_keys.merge(ref_attributes,on=['author_id','cohort_record_id'],how='left',validate='one_to_one',indicator=True)
assert rep['_merge'].eq('both').all()
rep_fields=['author_id','author_order_clean','n_authors_on_paper','position_from_list',
            'position_check_status','n_populated_institution_entries','affiliation_requires_review']
rep=rep[rep_fields].rename(columns={c:'reference_'+c for c in rep_fields if c!='author_id'})
cleaned=roster.merge(firstpub.drop(columns='reference_award_year'),on='author_id',validate='one_to_one').merge(rep,on='author_id',validate='one_to_one')
reference_detail={aid:json.dumps(g[[c for c in ['cohort_record_id','conference','paper_title','openalex_work_id',
                    'author_order_clean','n_authors_on_paper','position_from_list','institutions_json',
                    'raw_affiliation_strings_json'] if c in g]].astype(object).where(pd.notna(g),None).to_dict('records'),
                    ensure_ascii=False,default=lambda v:int(v) if hasattr(v,'item') else str(v))
                  for aid,g in ref_attributes.groupby('author_id')}
cleaned['reference_year_attribute_records_json']=cleaned.author_id.map(reference_detail)
any_aff_issue=ref_attributes.groupby('author_id').affiliation_requires_review.any()
any_pos_issue=ref_attributes.assign(issue=ref_attributes.position_check_status.ne('consistent_with_source_list')).groupby('author_id').issue.any()
cleaned['any_reference_paper_affiliation_requires_review']=cleaned.author_id.map(any_aff_issue)
cleaned['any_reference_paper_position_requires_review']=cleaned.author_id.map(any_pos_issue)
cleaned['author_attributes_require_review']=cleaned.first_pub_requires_review | cleaned.any_reference_paper_affiliation_requires_review | cleaned.any_reference_paper_position_requires_review
assert cleaned.author_id.is_unique and len(cleaned)==len(roster)
firstpub_review=firstpub.loc[firstpub.first_pub_requires_review].merge(roster[['author_id','author_name']],on='author_id',validate='one_to_one')
position_review=ref_attributes.loc[ref_attributes.position_check_status.ne('consistent_with_source_list')]
affiliation_review=ref_attributes.loc[ref_attributes.affiliation_requires_review]
summary=pd.DataFrame([
    ('authors_retained',len(cleaned)),
    ('junior_passed_basic_checks',int(cleaned.seniority_clean.eq('Junior').sum())),
    ('senior_passed_basic_checks',int(cleaned.seniority_clean.eq('Senior').sum())),
    ('seniority_unknown_requires_review',int(cleaned.seniority_clean.eq('Unknown').sum())),
    ('firstpub_review_authors',len(firstpub_review)),
    ('reference_paper_authorships',len(ref_attributes)),
    ('reference_paper_position_review_rows',len(position_review)),
    ('reference_paper_affiliation_review_rows',len(affiliation_review)),
    ('authors_with_any_attribute_review',int(cleaned.author_attributes_require_review.sum()))
],columns=['measure','value'])
display(summary)
display(cleaned[['author_id','author_name','reference_award_year','first_pub_year_profile',
                 'first_pub_year_cache','career_age_candidate','seniority_candidate','seniority_clean',
                 'first_pub_validation_status','reference_author_order_clean']].head(20))

,measure,value
0,authors_retained,2710
1,junior_passed_basic_checks,776
2,senior_passed_basic_checks,1723
3,seniority_unknown_requires_review,211
4,firstpub_review_authors,211
5,reference_paper_authorships,2737
6,reference_paper_position_review_rows,0
7,reference_paper_affiliation_review_rows,164
8,authors_with_any_attribute_review,359


,author_id,author_name,reference_award_year,first_pub_year_profile,first_pub_year_cache,career_age_candidate,seniority_candidate,seniority_clean,first_pub_validation_status,reference_author_order_clean
0,A5000009841,Ana Klimovic,2014,2013,2013,1,Junior,Junior,passed_basic_cache_checks_not_identity_validated,3
1,A5000102893,David Akers,2009,2002,2002,7,Senior,Senior,passed_basic_cache_checks_not_identity_validated,1
2,A5000107991,Michael Vrable,2008,2005,2005,3,Junior,Junior,passed_basic_cache_checks_not_identity_validated,3
3,A5000163896,Elisa D. Mekler,2016,2012,2012,4,Junior,Junior,passed_basic_cache_checks_not_identity_validated,1
4,A5000167011,I. Scott MacKenzie,2008,1988,1988,20,Senior,Senior,passed_basic_cache_checks_not_identity_validated,4
5,A5000170074,Alexandru Agache,2017,2011,2011,6,Senior,Senior,passed_basic_cache_checks_not_identity_validated,3
6,A5000245150,Alexander J. Smola,2014,1997,1997,17,Senior,Senior,passed_basic_cache_checks_not_identity_validated,4
7,A5000246567,Matthew L. Lee,2014,2014,2014,0,Junior,Junior,passed_basic_cache_checks_not_identity_validated,1
8,A5000294737,Philipp Schwartenbeck,2016,2013,2013,3,Junior,Junior,passed_basic_cache_checks_not_identity_validated,4
9,A5000308912,Jaechang Nam,2013,2009,2009,4,Junior,Junior,passed_basic_cache_checks_not_identity_validated,2


## 5. Export to a separate derived directory

The notebook refuses to overwrite a different existing result. On a rerun with identical inputs/settings it verifies existing output hashes. To deliberately rerun after data repairs, change the version rather than edit frozen data.

`firstpub_review.csv` is the next manual/targeted-recollection queue. Do not turn its candidate values into accepted labels just to eliminate Unknowns. Live earliest-work retrieval can be added as a subsequent read-only collection notebook, retaining evidence and respecting API limits.

In [7]:
settings={'transform_version':TRANSFORM_VERSION,'use_works_cache':USE_WORKS_CACHE,
          'early_year_review_cutoff':EARLY_YEAR_REVIEW_CUTOFF,'max_career_age_review':MAX_CAREER_AGE_REVIEW,
          'junior_cutoff':JUNIOR_CUTOFF,'cache_status':cache_status}
identity={'input_hashes':input_hashes,'settings':settings}
if WRITE_OUTPUTS:
    if TARGET.exists():
        old=json.loads((TARGET/'manifest.json').read_text(encoding='utf-8'))
        if old['identity']!=identity:raise RuntimeError('Inputs/settings changed. Choose a new OUTPUT_VERSION after review.')
        for name,h in old['output_sha256'].items():
            if not (TARGET/name).is_file() or digest(TARGET/name)!=h:raise RuntimeError('Existing output changed: '+name)
        print('Verified existing step-3 outputs; nothing overwritten:',TARGET)
    else:
        TARGET.parent.mkdir(parents=True,exist_ok=True)
        staging=Path(tempfile.mkdtemp(prefix='.'+OUTPUT_VERSION+'_',dir=TARGET.parent))
        try:
            exports={'author_attributes.csv':cleaned,'firstpub_evidence.csv':firstpub,
                     'earliest_cached_works.csv':earliest_evidence,'reference_paper_attributes.csv':ref_attributes,
                     'reference_paper_affiliations.csv':affiliations,'firstpub_review.csv':firstpub_review,
                     'position_review.csv':position_review,'affiliation_review.csv':affiliation_review,
                     'step3_summary.csv':summary}
            for name,df in exports.items():df.to_csv(staging/name,index=False)
            notes=('Step 3: provisional author attribute audit. One author row per identified ID. '
                   'seniority_clean is Unknown for flagged cases; passing basic cache checks does not verify identity '
                   'or earliest-career completeness. Cache earliest dates and existing profile dates are preserved separately. '
                   'Paper affiliation and order are original metadata, not independently PDF/employment-verified. '
                   'All reference-year papers and affiliations remain in long outputs. No APIs or source edits. '
                   'Use firstpub_review.csv for manual review or targeted recollection.\n')
            (staging/'README.md').write_text(notes,encoding='utf-8')
            hashes={p.name:digest(p) for p in sorted(staging.iterdir()) if p.is_file()}
            m={'created_at_utc':datetime.now(timezone.utc).isoformat(),'identity':identity,
               'output_sha256':hashes,'external_api_calls':False,'upstream_files_modified':False,
               'definitions':{'Junior':'0 <= reference year - candidate first year <= 5; must pass review checks',
                              'Senior':'career age > 5; must pass review checks',
                              'Unknown':'missing/unverified/review-flagged candidate; never treated as Senior',
                              'reference_author_order_clean':'1-based original paper authorship index, not a rerank of identified authors'}}
            (staging/'manifest.json').write_text(json.dumps(m,indent=2),encoding='utf-8')
            if TARGET.exists():raise FileExistsError('Destination appeared during export; refusing overwrite.')
            staging.rename(TARGET)
        finally:
            if staging.exists():shutil.rmtree(staging)
        print('Created step-3 audit outputs:',TARGET)
else:
    print('Preview only; no outputs written.')

Created step-3 audit outputs: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\derived\author_attributes_v1
